# Machine Learning Project: Using Sleep/Health Data (100k Rows) to assess my own WHOOP Data 

### Dashiell Kwan, 2026

#### Intro: Data Cleaning

In [1]:
import pandas as pd

In [2]:
sleep_data = pd.read_csv('/Users/dashkwan/Desktop/Programming/sleep_whoop/sleep_health_dataset_100k.csv')
sleep_data.columns

Index(['person_id', 'age', 'gender', 'occupation', 'bmi', 'country',
       'sleep_duration_hrs', 'sleep_quality_score', 'rem_percentage',
       'deep_sleep_percentage', 'sleep_latency_mins',
       'wake_episodes_per_night', 'caffeine_mg_before_bed',
       'alcohol_units_before_bed', 'screen_time_before_bed_mins',
       'exercise_day', 'steps_that_day', 'nap_duration_mins', 'stress_score',
       'work_hours_that_day', 'chronotype', 'mental_health_condition',
       'heart_rate_resting_bpm', 'sleep_aid_used', 'shift_work',
       'room_temperature_celsius', 'weekend_sleep_diff_hrs', 'season',
       'day_type', 'cognitive_performance_score', 'sleep_disorder_risk',
       'felt_rested'],
      dtype='str')

In [3]:
sleep_data = sleep_data.drop(columns=['person_id', 'occupation', 'country', 'season', 'chronotype', 'day_type', 'mental_health_condition', 'room_temperature_celsius', 'sleep_disorder_risk'])

In [4]:
sleep_data.head()

,age,gender,bmi,sleep_duration_hrs,sleep_quality_score,rem_percentage,deep_sleep_percentage,sleep_latency_mins,wake_episodes_per_night,caffeine_mg_before_bed,...,steps_that_day,nap_duration_mins,stress_score,work_hours_that_day,heart_rate_resting_bpm,sleep_aid_used,shift_work,weekend_sleep_diff_hrs,cognitive_performance_score,felt_rested
0,29,Female,25.7,6.19,6.6,22.5,19.3,16,3,0,...,6592,0,4.4,10.7,63,0,0,1.84,73.4,0
1,55,Female,22.0,8.32,6.9,26.9,14.9,17,4,0,...,10111,8,4.0,3.0,52,1,0,0.13,99.4,1
2,42,Male,25.0,3.74,1.0,20.2,16.2,26,4,0,...,9222,28,7.8,3.6,72,0,1,1.67,2.5,0
3,37,Female,29.5,6.79,6.4,17.7,17.7,13,4,0,...,9190,40,4.9,6.7,71,0,0,2.37,67.8,0
4,23,Male,23.6,5.02,3.2,23.3,18.3,30,5,40,...,4273,0,7.4,10.4,71,0,0,1.26,38.1,0


In [5]:
print(sleep_data.shape)
print(sleep_data.isna().sum())
print(sleep_data.duplicated().sum())
print(sleep_data['felt_rested'].value_counts())

(100000, 23)
age                            0
gender                         0
bmi                            0
sleep_duration_hrs             0
sleep_quality_score            0
rem_percentage                 0
deep_sleep_percentage          0
sleep_latency_mins             0
wake_episodes_per_night        0
caffeine_mg_before_bed         0
alcohol_units_before_bed       0
screen_time_before_bed_mins    0
exercise_day                   0
steps_that_day                 0
nap_duration_mins              0
stress_score                   0
work_hours_that_day            0
heart_rate_resting_bpm         0
sleep_aid_used                 0
shift_work                     0
weekend_sleep_diff_hrs         0
cognitive_performance_score    0
felt_rested                    0
dtype: int64
0
felt_rested
0    60988
1    39012
Name: count, dtype: int64


In [6]:
sleep_data['exercise_day'].unique()

array([0, 1])

In [7]:
features = ['age', 'bmi', 'sleep_duration_hrs', 'caffeine_mg_before_bed', 
            'alcohol_units_before_bed', 
            'steps_that_day', 'nap_duration_mins', 'stress_score', 
            'work_hours_that_day', 'heart_rate_resting_bpm', 
            'weekend_sleep_diff_hrs', 'exercise_day']

X = sleep_data[features]
y = sleep_data['felt_rested']

X.dtypes

age                           int64
bmi                         float64
sleep_duration_hrs          float64
caffeine_mg_before_bed        int64
alcohol_units_before_bed    float64
steps_that_day                int64
nap_duration_mins             int64
stress_score                float64
work_hours_that_day         float64
heart_rate_resting_bpm        int64
weekend_sleep_diff_hrs      float64
exercise_day                  int64
dtype: object

#### Target: y (felt rested)

## Train/Test Split

- Using stratify=y because of the imbalance between Sleep Quality labels
- Also using test_size = 0.2 (20%) as a standard

In [8]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [9]:
y_train.value_counts(normalize=True)
y_test.value_counts(normalize=True)

felt_rested
0    0.6099
1    0.3901
Name: proportion, dtype: float64

In [10]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

log_reg_pipeline = make_pipeline(
    StandardScaler(),
    LogisticRegression()
)

log_reg_pipeline.fit(X_train, y_train)
log_reg_pipeline.score(X_test, y_test)

0.71635

71%!!!??? Sheesh. But this is good, the old one had like 95% on the first run because we had such little data. 

In [11]:
from sklearn.metrics import confusion_matrix, classification_report

y_pred = log_reg_pipeline.predict(X_test)
print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[10155  2043]
 [ 3630  4172]]
              precision    recall  f1-score   support

           0       0.74      0.83      0.78     12198
           1       0.67      0.53      0.60      7802

    accuracy                           0.72     20000
   macro avg       0.70      0.68      0.69     20000
weighted avg       0.71      0.72      0.71     20000



The model is worse at catching "felt rested" than not. Model only caught 53% of people who actually felt rested based on this data. The model leans more towards not rested (recall is 0.83). 

In [12]:
print(log_reg_pipeline.named_steps['logisticregression'].coef_)

[[-0.03846251 -0.12233925  0.91047318 -0.0561235  -0.12483993  0.00338788
  -0.00946468 -0.51120546  0.01404497 -0.01203776  0.00325633  0.02622088]]


Strongest observations here: sleep duration (hrs) has strongest coefficient: +0.927. Wake episodes per night has -0.442, stress_score has -0.432, sleep latency has -0.153 and alcohol before bed has -0.071. Everything else lower. The 3 strongest variables are doing most of the work

In [13]:
log_reg_balanced = make_pipeline(StandardScaler(), LogisticRegression(class_weight='balanced'))
log_reg_balanced.fit(X_train, y_train)
print(classification_report(y_test, log_reg_balanced.predict(X_test)))

              precision    recall  f1-score   support

           0       0.80      0.72      0.76     12198
           1       0.62      0.71      0.66      7802

    accuracy                           0.72     20000
   macro avg       0.71      0.72      0.71     20000
weighted avg       0.73      0.72      0.72     20000



Claude:
This is a genuinely good thing to have in your project, because it's a real judgment call, not just a number to report:
Which version is "better" depends on what mistake matters more in context — and that's worth actually deciding and justifying, not leaving ambiguous:

If this were a wellness app nudging someone to rest more, missing "you're actually fine" (false negative on class 0) is low-stakes — a little unnecessary caution. Missing "you're not actually rested" (false negative on class 1, the original model's weakness) is worse, because it tells someone they're fine when they're not. By that logic, the balanced version is the better choice, since it specifically fixes that failure mode.
If instead you're optimizing for raw accuracy and don't care which class you're wrong about, they're identical (both 0.73).

I'd write this up as a deliberate choice: **"I evaluated both the default and class-weight-balanced logistic regression. Since a false 'you're rested' prediction is more consequential than the reverse in a wellness context, I selected the balanced model despite equal overall accuracy."**

## Random Forest

The comparison model. Random forest splits data based on thresholds per feature independently (e.g., "is sleep_duration_hrs > 6.5?") — the actual scale of the numbers doesn't affect how it splits. 

In [14]:
from sklearn.ensemble import RandomForestClassifier

rf_pipeline = make_pipeline(
    RandomForestClassifier(random_state=42)
)

rf_pipeline.fit(X_train, y_train)
rf_pipeline.score(X_test, y_test)

0.7229

In [15]:
y_pred_rf = rf_pipeline.predict(X_test)
print(confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

[[9704 2494]
 [3048 4754]]
              precision    recall  f1-score   support

           0       0.76      0.80      0.78     12198
           1       0.66      0.61      0.63      7802

    accuracy                           0.72     20000
   macro avg       0.71      0.70      0.70     20000
weighted avg       0.72      0.72      0.72     20000



Random Forest sits between the two logistic regression versions: recall on class 1 is 0.61 (better than default LR's 0.57, worse than balanced LR's 0.73), and class 0 recall is 0.81 (worse than default LR's 0.83, better than balanced LR's 0.73).

Consistency: three models all around 71-73%

In [16]:
rf_balanced = make_pipeline(RandomForestClassifier(random_state=42, class_weight='balanced'))
rf_balanced.fit(X_train, y_train)
print(classification_report(y_test, rf_balanced.predict(X_test)))

              precision    recall  f1-score   support

           0       0.79      0.73      0.76     12198
           1       0.62      0.69      0.66      7802

    accuracy                           0.72     20000
   macro avg       0.71      0.71      0.71     20000
weighted avg       0.72      0.72      0.72     20000



Checking feature importances, comparing the importance of each feature to the random forest

In [17]:
importances = pd.Series(rf_pipeline.named_steps['randomforestclassifier'].feature_importances_, index=features)
importances.sort_values(ascending=False)

sleep_duration_hrs          0.254453
stress_score                0.135415
work_hours_that_day         0.095601
steps_that_day              0.093892
weekend_sleep_diff_hrs      0.089921
bmi                         0.087265
age                         0.074131
heart_rate_resting_bpm      0.069180
nap_duration_mins           0.043831
caffeine_mg_before_bed      0.025879
alcohol_units_before_bed    0.018246
exercise_day                0.012186
dtype: float64

Random Forest and Logistic Regression agree where sleep duration and stress score are the two most important features, but they disagree elsewhere: like work hours that day. Wake episodes per night had a strong importance in LR but is basically 0 in random forest. 

Logistic Regression assumes a straight-line relationship between each feature and the outcome, so if a feature only matters at extremes, a linear coefficient would average it towards 0. But a tree-based model can split based on thresholds and catch the effect. 

## Model Comparison

| Model | Accuracy | Class 0 Precision | Class 0 Recall | Class 1 Precision | Class 1 Recall |
|---|---|---|---|---|---|
| Logistic Regression (default) | 0.716 | 0.74 | 0.83 | 0.67 | 0.53 |
| Logistic Regression (balanced) | 0.716 | 0.80 | 0.72 | 0.62 | 0.71 |
| Random Forest (default) | 0.723 | 0.76 | 0.80 | 0.66 | 0.61 |
| Random Forest (balanced) | 0.718 | 0.79 | 0.73 | 0.62 | 0.69 |

All four models land within a percentage point of each other on accuracy (71.6%-72.3%), reinforcing that this ceiling reflects the limits of the feature set rather than any one model's weakness — a result that held even after trimming from 15 to 12 features (dropping `wake_episodes_per_night`, `sleep_latency_mins`, and `screen_time_before_bed_mins`, which had no reliable wearable-data equivalent for the personal demo). As before, the more meaningful lever was `class_weight='balanced'`, which traded a modest amount of class 0 recall for a substantial improvement in class 1 recall, rather than improving performance outright.

**Selected model: Logistic Regression (balanced).** In a wellness context, a false "you're rested" prediction (missing a true class-1 case) is more consequential than the reverse — it risks pushing into a demanding day or workout without knowing the body needs recovery. Logistic regression (balanced) achieves the strongest class 1 recall (0.71) among all four variants while remaining the most interpretable model.

**Feature importance agreement:** both models agree `sleep_duration_hrs` and `stress_score` are by far the two strongest predictors. They diverge on secondary features — Random Forest ranked `work_hours_that_day`, `steps_that_day`, `weekend_sleep_diff_hrs`, and `bmi` notably higher than Logistic Regression's coefficients suggested, consistent with these features having a nonlinear or threshold-based relationship with rest that a linear model underweights.

# Personalization: Mapping Whoop Data

In [18]:
workouts = pd.read_csv('/Users/dashkwan/Desktop/Programming/sleep_whoop/whoop_data/workouts.csv')
sleeps = pd.read_csv('/Users/dashkwan/Desktop/Programming/sleep_whoop/whoop_data/sleeps.csv')
cycles = pd.read_csv('/Users/dashkwan/Desktop/Programming/sleep_whoop/whoop_data/physiological_cycles.csv')
entries = pd.read_csv('/Users/dashkwan/Desktop/Programming/sleep_whoop/whoop_data/journal_entries.csv')

In [19]:
workouts['Cycle start time'].duplicated().sum()

np.int64(144)

In [20]:
workouts_daily = workouts.groupby('Cycle start time').agg(
    workout_count=('Activity name', 'count'),
    total_duration_min=('Duration (min)', 'sum'),
    max_activity_strain=('Activity Strain', 'max'),
    total_energy_burned=('Energy burned (cal)', 'sum'),
    avg_hr=('Average HR (bpm)', 'mean')
).reset_index()

In [21]:
entries

,Cycle start time,Cycle end time,Cycle timezone,Question text,Answered yes,Notes
0,2026-07-12 01:21:09,NaN,UTC-04:00,Have any alcoholic drinks?,False,NaN
1,2026-07-12 01:21:09,NaN,UTC-04:00,Consumed caffeine?,False,NaN
2,2026-07-12 01:21:09,NaN,UTC-04:00,Ate food close to bedtime?,True,NaN
3,2026-07-12 01:21:09,NaN,UTC-04:00,Read (non-screened device) while in bed?,False,NaN
4,2026-07-11 00:54:22,2026-07-12 01:21:09,UTC-04:00,Have any alcoholic drinks?,False,NaN
...,...,...,...,...,...,...
687,2026-01-02 01:09:57,2026-01-03 01:41:42,UTC-05:00,Read (non-screened device) while in bed?,False,NaN
688,2026-01-01 02:13:00,2026-01-02 01:09:57,UTC-05:00,Have any alcoholic drinks?,True,NaN
689,2026-01-01 02:13:00,2026-01-02 01:09:57,UTC-05:00,Consumed caffeine?,False,NaN
690,2026-01-01 02:13:00,2026-01-02 01:09:57,UTC-05:00,Ate food close to bedtime?,True,NaN


In [22]:
print(sleeps['Cycle start time'].duplicated().sum())
print(cycles['Cycle start time'].duplicated().sum())
print(entries['Cycle start time'].duplicated().sum())

0
0
519


Fixing Entries table so that it doesn't have duplicate days (had 4 rows per day)

In [23]:
entries_wide = entries.pivot_table(
    index='Cycle start time',
    columns='Question text',
    values='Answered yes',
    aggfunc='first'
).reset_index()

In [24]:
entries_wide

Question text,Cycle start time,Ate food close to bedtime?,Consumed caffeine?,Have any alcoholic drinks?,Read (non-screened device) while in bed?
0,2026-01-01 02:13:00,True,False,True,False
1,2026-01-02 01:09:57,False,False,True,False
2,2026-01-03 01:41:42,False,False,True,False
3,2026-01-04 03:03:44,True,False,True,False
4,2026-01-05 01:20:05,True,False,False,False
...,...,...,...,...,...
168,2026-07-07 23:33:13,False,False,False,False
169,2026-07-08 23:39:06,False,False,True,False
170,2026-07-10 00:23:51,False,False,False,False
171,2026-07-11 00:54:22,True,False,False,False


Doing proper merges so that no days are left behind

In [25]:
whoop_data = pd.merge(sleeps, cycles, on='Cycle start time', how='left')
whoop_data = pd.merge(whoop_data, workouts_daily, on='Cycle start time', how='left')
whoop_data = pd.merge(whoop_data, entries_wide, on='Cycle start time', how='left')

In [26]:
whoop_data

,Cycle start time,Cycle end time_x,Cycle timezone_x,Sleep onset_x,Wake onset_x,Sleep performance %_x,Respiratory rate (rpm)_x,Asleep duration (min)_x,In bed duration (min)_x,Light sleep duration (min)_x,...,Sleep consistency %_y,workout_count,total_duration_min,max_activity_strain,total_energy_burned,avg_hr,Ate food close to bedtime?,Consumed caffeine?,Have any alcoholic drinks?,Read (non-screened device) while in bed?
0,2026-07-12 01:21:09,NaN,UTC-04:00,2026-07-12 01:21:09,2026-07-12 10:50:36,83,16.6,536,569,251,...,62.0,NaN,NaN,NaN,NaN,NaN,True,False,False,False
1,2026-07-11 00:54:22,2026-07-12 01:21:09,UTC-04:00,2026-07-11 00:54:22,2026-07-11 09:37:49,82,16.8,497,523,259,...,67.0,1.0,36.0,8.5,305.0,129.000000,True,False,False,False
2,2026-07-10 00:23:51,2026-07-11 00:54:22,UTC-04:00,2026-07-10 00:23:51,2026-07-10 07:59:40,84,16.8,442,455,231,...,82.0,2.0,58.0,5.8,281.0,114.000000,False,False,False,False
3,2026-07-08 23:39:06,2026-07-10 00:23:51,UTC-04:00,2026-07-08 23:39:06,2026-07-09 07:32:15,89,16.8,455,473,218,...,88.0,4.0,151.0,12.9,946.0,119.250000,False,False,True,False
4,2026-07-07 23:33:13,2026-07-08 23:39:06,UTC-04:00,2026-07-07 23:33:13,2026-07-08 07:26:11,87,17.1,455,472,230,...,83.0,1.0,36.0,10.1,317.0,131.000000,False,False,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
190,2026-01-03 01:41:42,2026-01-04 03:03:44,UTC-05:00,2026-01-03 01:41:42,2026-01-03 09:28:47,93,17.3,449,467,196,...,92.0,3.0,45.0,5.8,272.0,114.666667,False,False,True,False
191,2026-01-02 01:09:57,2026-01-03 01:41:42,UTC-05:00,2026-01-02 01:09:57,2026-01-02 09:37:36,96,17.0,496,507,260,...,91.0,NaN,NaN,NaN,NaN,NaN,False,False,True,False
192,2026-01-01 02:13:00,2026-01-02 01:09:57,UTC-05:00,2026-01-01 02:13:00,2026-01-01 09:35:00,88,16.9,419,442,193,...,85.0,NaN,NaN,NaN,NaN,NaN,True,False,True,False
193,2025-12-31 01:13:46,2026-01-01 02:13:00,UTC-05:00,2025-12-31 01:13:46,2025-12-31 09:19:31,100,16.8,468,485,222,...,NaN,1.0,16.0,5.1,80.0,114.000000,NaN,NaN,NaN,NaN


In [27]:
whoop_data['workout_count'] = whoop_data['workout_count'].fillna(0)
whoop_data['total_duration_min'] = whoop_data['total_duration_min'].fillna(0)
whoop_data['max_activity_strain'] = whoop_data['max_activity_strain'].fillna(0)
whoop_data['total_energy_burned'] = whoop_data['total_energy_burned'].fillna(0)

In [28]:
whoop_data.columns.to_list()

['Cycle start time',
 'Cycle end time_x',
 'Cycle timezone_x',
 'Sleep onset_x',
 'Wake onset_x',
 'Sleep performance %_x',
 'Respiratory rate (rpm)_x',
 'Asleep duration (min)_x',
 'In bed duration (min)_x',
 'Light sleep duration (min)_x',
 'Deep (SWS) duration (min)_x',
 'REM duration (min)_x',
 'Awake duration (min)_x',
 'Sleep need (min)_x',
 'Sleep debt (min)_x',
 'Sleep efficiency %_x',
 'Sleep consistency %_x',
 'Nap',
 'Cycle end time_y',
 'Cycle timezone_y',
 'Recovery score %',
 'Resting heart rate (bpm)',
 'Heart rate variability (ms)',
 'Skin temp (celsius)',
 'Blood oxygen %',
 'Day Strain',
 'Energy burned (cal)',
 'Max HR (bpm)',
 'Average HR (bpm)',
 'Sleep onset_y',
 'Wake onset_y',
 'Sleep performance %_y',
 'Respiratory rate (rpm)_y',
 'Asleep duration (min)_y',
 'In bed duration (min)_y',
 'Light sleep duration (min)_y',
 'Deep (SWS) duration (min)_y',
 'REM duration (min)_y',
 'Awake duration (min)_y',
 'Sleep need (min)_y',
 'Sleep debt (min)_y',
 'Sleep efficien

In [29]:
(whoop_data['Sleep performance %_x'] == whoop_data['Sleep performance %_y']).all()

np.True_

#### Drop redundant, unnecessary columns

In [30]:
whoop_data = whoop_data.drop(columns=[
    'Cycle end time_x', 'Cycle timezone_x', 'Sleep onset_x', 'Wake onset_x',
    'Respiratory rate (rpm)_x', 'In bed duration (min)_x', 'Light sleep duration (min)_x',
    'Deep (SWS) duration (min)_x', 'REM duration (min)_x', 'Sleep need (min)_x',
    'Sleep debt (min)_x', 'Sleep efficiency %_x', 'Sleep consistency %_x',
    'Cycle end time_y', 'Cycle timezone_y', 'Sleep onset_y', 'Wake onset_y',
    'Sleep performance %_y', 'Respiratory rate (rpm)_y', 'Asleep duration (min)_y',
    'In bed duration (min)_y', 'Light sleep duration (min)_y', 'Deep (SWS) duration (min)_y',
    'REM duration (min)_y', 'Awake duration (min)_y', 'Sleep need (min)_y',
    'Sleep debt (min)_y', 'Sleep efficiency %_y', 'Sleep consistency %_y',
    'Skin temp (celsius)', 'Blood oxygen %', 'Day Strain', 'Max HR (bpm)',
    'total_energy_burned', 'avg_hr', 'total_duration_min', 'max_activity_strain'
])

In [31]:
whoop_data = whoop_data.rename(columns={
    'Asleep duration (min)_x': 'sleep_duration_min',
    'Resting heart rate (bpm)': 'heart_rate_resting_bpm',
    'Nap': 'nap_duration_mins',
    'Awake duration (min)_x': 'awake_duration_min'
})

whoop_data['sleep_duration_hrs'] = whoop_data['sleep_duration_min'] / 60
whoop_data['exercise_day'] = (whoop_data['workout_count'] > 0).astype(int)
whoop_data['stress_score_proxy'] = 100 - whoop_data['Recovery score %']
whoop_data['caffeine_flag'] = whoop_data['Consumed caffeine?'].astype(float)
whoop_data['alcohol_flag'] = whoop_data['Have any alcoholic drinks?'].astype(float)

In [33]:
whoop_data['Cycle start time'] = pd.to_datetime(whoop_data['Cycle start time'])

whoop_data['is_weekend'] = whoop_data['Cycle start time'].dt.dayofweek >= 5
weekday_avg = whoop_data.loc[~whoop_data['is_weekend'], 'sleep_duration_hrs'].mean()
weekend_avg = whoop_data.loc[whoop_data['is_weekend'], 'sleep_duration_hrs'].mean()
whoop_data['weekend_sleep_diff_hrs'] = weekend_avg - weekday_avg

In [34]:
whoop_data['Cycle start time'] = pd.to_datetime(whoop_data['Cycle start time'])
demo_data = whoop_data[whoop_data['Cycle start time'] >= '2026-07-01'].copy()
print(demo_data.shape)

(11, 22)


Manual Fill

In [35]:
work_hours_lookup = {
    '2026-07-01': 8,
    '2026-07-02': 8,
    '2026-07-03': 0,
    '2026-07-04': 0,
    '2026-07-05': 0,
    '2026-07-06': 8,
    '2026-07-07': 8,
    '2026-07-08': 8,
    '2026-07-09': 8,
    '2026-07-10': 8,
    '2026-07-11': 0,
    '2026-07-12': 4
}

demo_data['date_str'] = demo_data['Cycle start time'].dt.strftime('%Y-%m-%d')
demo_data['work_hours_that_day'] = demo_data['date_str'].map(work_hours_lookup)

In [36]:
steps_lookup = {
    '2026-07-01': 14636,
    '2026-07-02': 14661,
    '2026-07-03': 16492,
    '2026-07-04': 14412,
    '2026-07-05': 10178,
    '2026-07-06': 15063,
    '2026-07-07': 10355,
    '2026-07-08': 8878,
    '2026-07-09': 11996,
    '2026-07-10': 12928,
    '2026-07-11': 8465,
    '2026-07-12': 8745
}

demo_data['steps_that_day'] = demo_data['date_str'].map(steps_lookup)

In [37]:
demo_data['date_str'].tolist()

['2026-07-12',
 '2026-07-11',
 '2026-07-10',
 '2026-07-08',
 '2026-07-07',
 '2026-07-06',
 '2026-07-05',
 '2026-07-05',
 '2026-07-04',
 '2026-07-03',
 '2026-07-01']

### Running the pipeline 

In [46]:
demo_data = demo_data.rename(columns={
    'caffeine_flag': 'caffeine_mg_before_bed',
    'alcohol_flag': 'alcohol_units_before_bed',
    'stress_score_proxy': 'stress_score'
})

demo_data['age'] = 21   
demo_data['bmi'] = 23.6  

In [47]:
demo_data['caffeine_mg_before_bed'] = demo_data['caffeine_mg_before_bed'].fillna(0)
demo_data['alcohol_units_before_bed'] = demo_data['alcohol_units_before_bed'].fillna(0)

In [48]:
final_features = ['age', 'bmi', 'sleep_duration_hrs', 'caffeine_mg_before_bed', 
                   'alcohol_units_before_bed', 'steps_that_day', 'nap_duration_mins',
                   'stress_score', 'work_hours_that_day', 'heart_rate_resting_bpm',
                   'weekend_sleep_diff_hrs', 'exercise_day']

demo_X = demo_data[final_features]
demo_X.dtypes
demo_X.isna().sum()

age                         0
bmi                         0
sleep_duration_hrs          0
caffeine_mg_before_bed      0
alcohol_units_before_bed    0
steps_that_day              0
nap_duration_mins           0
stress_score                0
work_hours_that_day         0
heart_rate_resting_bpm      0
weekend_sleep_diff_hrs      0
exercise_day                0
dtype: int64

In [49]:
demo_data['predicted_rested'] = log_reg_balanced.predict(demo_X)
demo_data['predicted_prob_rested'] = log_reg_balanced.predict_proba(demo_X)[:, 1]

demo_data[['date_str', 'predicted_rested', 'predicted_prob_rested', 'Recovery score %']]

,date_str,predicted_rested,predicted_prob_rested,Recovery score %
0,2026-07-12,0,2.702468e-02,77.0
1,2026-07-11,0,5.180556e-04,66.0
2,2026-07-10,0,1.853710e-03,72.0
3,2026-07-08,0,3.687965e-03,74.0
4,2026-07-07,0,7.875304e-03,76.0
5,2026-07-06,1,7.365848e-01,94.0
6,2026-07-05,0,2.465021e-02,82.0
7,2026-07-05,0,1.397557e-07,44.0
8,2026-07-04,0,4.552621e-07,45.0
9,2026-07-03,0,2.218231e-06,51.0


In [50]:
demo_data[['date_str', 'predicted_prob_rested', 'Recovery score %']].sort_values('date_str')

,date_str,predicted_prob_rested,Recovery score %
10,2026-07-01,3.955136e-04,67.0
9,2026-07-03,2.218231e-06,51.0
8,2026-07-04,4.552621e-07,45.0
6,2026-07-05,2.465021e-02,82.0
7,2026-07-05,1.397557e-07,44.0
5,2026-07-06,7.365848e-01,94.0
4,2026-07-07,7.875304e-03,76.0
3,2026-07-08,3.687965e-03,74.0
2,2026-07-10,1.853710e-03,72.0
1,2026-07-11,5.180556e-04,66.0


Fixing format

In [51]:
pd.set_option('display.float_format', '{:.4f}'.format)
demo_data[['date_str', 'predicted_rested', 'predicted_prob_rested', 'Recovery score %']]

,date_str,predicted_rested,predicted_prob_rested,Recovery score %
0,2026-07-12,0,0.0270,77.0000
1,2026-07-11,0,0.0005,66.0000
2,2026-07-10,0,0.0019,72.0000
3,2026-07-08,0,0.0037,74.0000
4,2026-07-07,0,0.0079,76.0000
5,2026-07-06,1,0.7366,94.0000
6,2026-07-05,0,0.0247,82.0000
7,2026-07-05,0,0.0000,44.0000
8,2026-07-04,0,0.0000,45.0000
9,2026-07-03,0,0.0000,51.0000


Something is wrong...

In [60]:
print('sleep data vs demo data stress score\n')
print(sleep_data['stress_score'].describe())
print(demo_data['stress_score'].describe())

sleep data vs demo data stress score

count   100000.0000
mean         5.7333
std          1.6192
min          1.0000
25%          4.8000
50%          5.8000
75%          6.8000
max         10.0000
Name: stress_score, dtype: float64
count   11.0000
mean    32.0000
std     15.7099
min      6.0000
25%     23.5000
50%     28.0000
75%     41.5000
max     56.0000
Name: stress_score, dtype: float64


Fixing calculation scaling 

In [64]:
train_min = sleep_data['stress_score'].min()
train_max = sleep_data['stress_score'].max()

proxy_min = demo_data['stress_score'].min()
proxy_max = demo_data['stress_score'].max()

demo_data['stress_score'] = train_min + (demo_data['stress_score'] - proxy_min) / (proxy_max - proxy_min) * (train_max - train_min)

In [65]:
demo_X = demo_data[final_features]
demo_data['predicted_rested'] = log_reg_balanced.predict(demo_X)
demo_data['predicted_prob_rested'] = log_reg_balanced.predict_proba(demo_X)[:, 1]
demo_data[['date_str', 'predicted_rested', 'predicted_prob_rested', 'Recovery score %']]

,date_str,predicted_rested,predicted_prob_rested,Recovery score %
0,2026-07-12,1,0.9240,77.0000
1,2026-07-11,1,0.8041,66.0000
2,2026-07-10,1,0.7519,72.0000
3,2026-07-08,1,0.7810,74.0000
4,2026-07-07,1,0.8188,76.0000
5,2026-07-06,1,0.9330,94.0000
6,2026-07-05,1,0.7477,82.0000
7,2026-07-05,0,0.2661,44.0000
8,2026-07-04,0,0.4758,45.0000
9,2026-07-03,0,0.4768,51.0000


Model seems pretty accurate! Time to quantify

In [66]:
demo_data['whoop_rested'] = (demo_data['Recovery score %'] >= 67).astype(int)
demo_data['match'] = (demo_data['predicted_rested'] == demo_data['whoop_rested']).astype(int)

agreement_rate = demo_data['match'].mean()
print(agreement_rate)
demo_data[['date_str', 'predicted_rested', 'Recovery score %', 'whoop_rested', 'match']]

0.9090909090909091


,date_str,predicted_rested,Recovery score %,whoop_rested,match
0,2026-07-12,1,77.0000,1,1
1,2026-07-11,1,66.0000,0,0
2,2026-07-10,1,72.0000,1,1
3,2026-07-08,1,74.0000,1,1
4,2026-07-07,1,76.0000,1,1
5,2026-07-06,1,94.0000,1,1
6,2026-07-05,1,82.0000,1,1
7,2026-07-05,0,44.0000,0,1
8,2026-07-04,0,45.0000,0,1
9,2026-07-03,0,51.0000,0,1
